# Dragon DDict + MPI Data Sharing Tutorial

**Estimated time:** ~35-50 minutes  
**Format:** 5 exercises with demo code, a short coding task, and hidden solutions.

---

## Session goals

This tutorial introduces how to share data between MPI applications and
other processes using Dragon's DDict API.

You will practice how to:

- create and share DDict handles across process boundaries
- combine MPI-launched jobs with non-MPI parser/aggregator processes
- use atomic DDict updates for concurrent writers
- route rank output and persist parsed results in DDict
- pass DDict handles into MPI Python code

The progression is suitable for beginners and also highlights patterns
used by intermediate and advanced users in MPI workflows.

---

## Setup - run this first

Start Jupyter from a Dragon-enabled environment and run the next cell.

In [ ]:
import os
import re
import sys
import socket
from pathlib import Path

import dragon
import multiprocessing as mp
import dragon.native as dn

from dragon.data.ddict import DDict
from dragon.native.process import Process, ProcessTemplate, MSG_PIPE, MSG_DEVNULL
from dragon.native.process_group import ProcessGroup
from dragon.infrastructure.facts import PMIBackend

try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

print("Dragon + DDict ready")

DDict sharing pattern used throughout this notebook:

```python
d = DDict(1, 1, 64 * 1024 * 1024)
handle = d.serialize()
# ... pass handle to another process ...
d2 = DDict.attach(handle)
# use d2
d2.detach()
d.destroy()
```

---

## Exercise 1 - Share a DDict between standard Dragon processes

**Background:**

Before involving MPI, verify DDict sharing with regular Dragon processes.
A child receives a serialized DDict handle, attaches, writes data, then detaches.

Demo pattern:

```python
def worker(serialized_ddict):
    d = DDict.attach(serialized_ddict)
    d["child"] = "hello"
    d.detach()
```

**Your task:**

1. Write `writer_worker(serialized_ddict, rank)`
2. Worker should attach and set `d[f"rank_{rank}"] = rank * 10`
3. Launch 4 workers
4. In parent, print keys and values
5. Detach and destroy cleanly

In [ ]:
# -- Exercise 1 -- your code here -----------------------------------------------

def writer_worker(serialized_ddict, rank):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def writer_worker(serialized_ddict, rank):
    d = DDict.attach(serialized_ddict)
    d[f"rank_{rank}"] = rank * 10
    d.detach()

d = DDict(1, 1, 64 * 1024 * 1024)
serialized = d.serialize()

procs = [dn.Process(target=writer_worker, args=(serialized, r)) for r in range(4)]
for p in procs:
    p.start()
for p in procs:
    p.join()

for k in sorted(list(d.keys())):
    print(k, d[k])

d.detach()
d.destroy()
```

</details>

---

## Exercise 2 - Concurrent updates with `fetch_add`

**Background:**

When many ranks/processes update one counter, use `fetch_add` for atomic
increment semantics.

Demo pattern:

```python
d.fetch_add("total", 1)   # atomic increment
value = d.fetch_add("total", 0)  # atomic read of fetch_add-encoded value
```

**Your task:**

1. Write `counter_worker(serialized_ddict, nsteps)`
2. Worker should call `fetch_add("token_count", 1)` exactly `nsteps` times
3. Launch 8 workers with `nsteps=500`
4. Print final value using `fetch_add("token_count", 0)`
5. Verify expected total

In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------

def counter_worker(serialized_ddict, nsteps):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def counter_worker(serialized_ddict, nsteps):
    d = DDict.attach(serialized_ddict)
    for _ in range(nsteps):
        d.fetch_add("token_count", 1)
    d.detach()

d = DDict(1, 1, 64 * 1024 * 1024)
serialized = d.serialize()

nworkers = 8
nsteps = 500
procs = [dn.Process(target=counter_worker, args=(serialized, nsteps)) for _ in range(nworkers)]
for p in procs:
    p.start()
for p in procs:
    p.join()

final_val = d.fetch_add("token_count", 0)
print("final:", final_val)
print("expected:", nworkers * nsteps)

d.detach()
d.destroy()
```

</details>

---

## Exercise 3 - Launch MPI and parse rank output into DDict

**Background:**

A practical integration pattern is to launch an MPI executable with
ProcessGroup, read rank-0 stdout from `MSG_PIPE`, parse lines, and write
parsed results to DDict for downstream consumers.

Demo pattern:

```python
pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
pg.add_process(nproc=1, template=ProcessTemplate(target=exe, stdout=MSG_PIPE))
pg.add_process(nproc=n-1, template=ProcessTemplate(target=exe, stdout=MSG_DEVNULL))
```

**Your task:**

1. Implement `run_mpi_to_ddict(exe_path, total_ranks, serialized_ddict)`
2. Launch MPI with rank 0 piped and others DEVNULL
3. Read rank-0 lines and append them to `d["mpi_lines"]`
4. Store `d["mpi_done"] = True` when complete
5. Join and close the ProcessGroup

In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------

def run_mpi_to_ddict(exe_path, total_ranks, serialized_ddict):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_mpi_to_ddict(exe_path, total_ranks, serialized_ddict):
    d = DDict.attach(serialized_ddict)
    d["mpi_lines"] = []

    pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target=exe_path, args=(), cwd=str(Path(exe_path).parent), stdout=MSG_PIPE),
    )
    if total_ranks > 1:
        pg.add_process(
            nproc=total_ranks - 1,
            template=ProcessTemplate(target=exe_path, args=(), cwd=str(Path(exe_path).parent), stdout=MSG_DEVNULL),
        )

    pg.init()
    pg.start()

    lines = []
    for puid in pg.puids:
        child = Process(None, ident=puid)
        if child.stdout_conn:
            try:
                while True:
                    lines.append(child.stdout_conn.recv().strip())
            except EOFError:
                pass

    d["mpi_lines"] = lines
    d["mpi_done"] = True

    pg.join()
    pg.close()
    d.detach()

exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
d = DDict(1, 1, 128 * 1024 * 1024)
serialized = d.serialize()

if Path(exe).exists():
    run_mpi_to_ddict(exe, 4, serialized)
    print("done flag:", d["mpi_done"])
    print("num lines captured:", len(d["mpi_lines"]))
else:
    print("Build mpi_hello in ../dragon_native/mpi first")

d.detach()
d.destroy()
```

</details>

---

## Exercise 4 - Orchestrate MPI producer and Python consumer via DDict

**Background:**

A common workflow is MPI producer + non-MPI consumer. Producer writes raw
results into DDict, and consumer process aggregates or transforms them.

Demo pattern:

```python
producer writes d["mpi_lines"]
consumer reads d["mpi_lines"] and writes d["summary"]
```

**Your task:**

1. Write `consumer_worker(serialized_ddict)`
2. Consumer should wait until `mpi_done` is set
3. Count lines containing `rank` and write `d["rank_line_count"]`
4. Launch MPI producer (Exercise 3) and consumer concurrently
5. Print `rank_line_count` from parent

In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------

def consumer_worker(serialized_ddict):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def consumer_worker(serialized_ddict):
    d = DDict.attach(serialized_ddict)
    while not d.get("mpi_done", False):
        pass
    lines = d.get("mpi_lines", [])
    count = sum(1 for line in lines if "rank" in line.lower())
    d["rank_line_count"] = count
    d.detach()

exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
d = DDict(1, 1, 128 * 1024 * 1024)
serialized = d.serialize()

if Path(exe).exists():
    producer = dn.Process(target=run_mpi_to_ddict, args=(exe, 4, serialized))
    consumer = dn.Process(target=consumer_worker, args=(serialized,))

    producer.start()
    consumer.start()

producer.join()
consumer.join()

print("rank_line_count:", d.get("rank_line_count", 0))
else:
    print("Build mpi_hello in ../dragon_native/mpi first")

d.detach()
d.destroy()
```

</details>

---

## Exercise 5 - Advanced: Access DDict directly from MPI Python ranks

**Background:**

Advanced users often want MPI ranks to attach to DDict directly and write
results by rank. This is typically done by passing a serialized DDict handle
via environment variable or command-line argument.

Demo pattern (inside MPI rank code):

```python
serialized = os.environ["DRAGON_DDICT_SER"]
d = DDict.attach(serialized)
d[f"rank_{rank}"] = payload
d.detach()
```

**Your task:**

1. Write a helper script file `mpi_ddict_rank_writer.py` from Python
2. Script should use `mpi4py` to get `rank` and write one key per rank to DDict
3. Launch script with ProcessGroup as an MPI job (`pmi=PMIBackend.CRAY`)
4. Pass serialized DDict handle in env var `DRAGON_DDICT_SER`
5. After completion, print all `rank_*` keys from parent

In [ ]:
# -- Exercise 5 -- your code here -----------------------------------------------

# Hint: use Path("mpi_ddict_rank_writer.py").write_text(...)

<details>
<summary><b>▶ Show Solution</b></summary>

```python
script_path = Path("mpi_ddict_rank_writer.py")
script_path.write_text(
    """
import os
from dragon.data.ddict import DDict
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

serialized = os.environ["DRAGON_DDICT_SER"]
d = DDict.attach(serialized)
d[f"rank_{rank}"] = {"rank": rank, "size": size}
d.detach()
""".strip() + "\n",
    encoding="utf-8",
)

d = DDict(1, 1, 128 * 1024 * 1024)
serialized = d.serialize()

env = os.environ.copy()
env["DRAGON_DDICT_SER"] = serialized

pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
pg.add_process(
    nproc=4,
    template=ProcessTemplate(
        target=sys.executable,
        args=(str(script_path),),
        env=env,
        cwd=os.getcwd(),
        stdout=MSG_DEVNULL,
    ),
)

try:
    pg.init()
    pg.start()
    pg.join()
    for k in sorted([key for key in d.keys() if str(key).startswith("rank_")]):
        print(k, d[k])
except Exception as ex:
    print("MPI Python example requires mpi4py-enabled environment:", ex)
finally:
    pg.close()
    d.detach()
    d.destroy()
```

</details>

---

## Summary

You used DDict as a shared data plane between MPI and non-MPI processes,
from simple handle passing to rank-level MPI writes.

| Concept | API |
|---|---|
| Create DDict | `DDict(managers_per_node, n_nodes, total_mem)` |
| Share handle | `d.serialize()` |
| Attach in another process | `DDict.attach(serialized_handle)` |
| Atomic update | `d.fetch_add(key, value)` |
| MPI orchestration | `ProcessGroup(..., pmi=PMIBackend.CRAY)` |
| Rank-0 output parsing | `stdout=MSG_PIPE` + `stdout_conn.recv()` |
| Quiet non-head ranks | `stdout=MSG_DEVNULL` |
| Cleanup | `detach()`, `destroy()`, `close()` |

### Next steps

- Replace `mpi_hello` with your application and parse domain-specific output.
- Store structured metrics in DDict (latency, rank placement, per-step stats).
- Combine DDict with Policies to co-locate parsers and consumers by node.